In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
sample_sub = pd.read_csv('../data/raw/sample_submission.csv')

print("train:", train.shape)
print("test:", test.shape)
print("sample_submission:", sample_sub.shape)

train: (1226237, 5)
test: (714688, 5)
sample_submission: (714688, 2)


In [6]:
print(f"Sütunların veri tipi: \n{train.dtypes}")
train.head()

Sütunların veri tipi: 
tanim           str
guc           int64
tarih           str
tuketim     float64
lokasyon        str
dtype: object


,tanim,guc,tarih,tuketim,lokasyon
0,70122340,1000,2025-01-01,5789.76,İZMİR>METROPOL>KARABAĞLAR
1,70142184,1250,2025-01-01,602.40,İZMİR>METROPOL>KARABAĞLAR
2,77540173,50,2025-01-01,19.92,MANİSA>GÖRDES
3,77540175,100,2025-01-01,25.64,MANİSA>GÖRDES
4,77540199,50,2025-01-01,10.98,MANİSA>GÖRDES


In [11]:
print(f"Train eksik değerler: \n{train.isna().sum()} \n")

print(f"Test eksik değerler: \n{test.isna().sum()}")

Train eksik değerler: 
tanim       0
guc         0
tarih       0
tuketim     0
lokasyon    0
dtype: int64 

Test eksik değerler: 
id          0
tanim       0
guc         0
tarih       0
lokasyon    0
dtype: int64


In [ ]:
#train ve test tarihleri datetime'a çevir (coldstart kontrolü)
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih'] = pd.to_datetime(test['tarih'])

print("Train tarih aralığı:", train['tarih'].min(), "-", train['tarih'].max())
print("Test tarih aralığı:", test['tarih'].min(), "-", test['tarih'].max())

train_tanim = set(train['tanim'].unique())
test_tanim = set(test['tanim'].unique())

print("Train'deki unique trafo sayısı:", len(train_tanim)) 
print("Test'teki unique trafo sayısı:", len(test_tanim))
print("Test'te olup train'de olmayan trafo sayısı:", len(test_tanim - train_tanim))
print("Train'de olup test'te olmayan trafo sayısı:", len(train_tanim - test_tanim))


#!! Bulgu: Test'teki 7036 trafonun 2024'ü train'de hiç yok. Yani test setinin neredeyse üçte biri cold start -> bu trafolar için hiç geçmiş tüketim verimiz olmayacak.

Train tarih aralığı: 2025-01-01 00:00:00 - 2026-03-31 00:00:00
Test tarih aralığı: 2026-04-01 00:00:00 - 2026-07-31 00:00:00
Train'deki unique trafo sayısı: 5344
Test'teki unique trafo sayısı: 7036
Test'te olup train'de olmayan trafo sayısı: 2024
Train'de olup test'te olmayan trafo sayısı: 332


In [ ]:
new_trafo = test_tanim - train_tanim

test_new = test[test['tanim'].isin(new_trafo)]
test_known = test[test['tanim'].isin(train_tanim)]

print("Yeni trafolar - guc istatistikleri: \n", test_new['guc'].describe())

print("\nBilinen trafolar - guc istatistikleri: \n", test_known['guc'].describe())

print("\nYeni trafolar - en sık lokasyonlar: \n", test_new['lokasyon'].value_counts().head(13))

print("\nBilinen trafolar - en sık lokasyonlar: \n", test_known['lokasyon'].value_counts().head(13))

## Bulgu: guc dağılımı yeni ve bilinen trafolarda benzer ->Mean: 676 vs 683 / Lokasyonlar da çok uzak değil (genellenebilir?)


Yeni trafolar - guc istatistikleri: 
 count    158369.000000
mean        676.713890
std         870.249229
min          40.000000
25%         250.000000
50%         630.000000
75%        1000.000000
max       30930.000000
Name: guc, dtype: float64

Bilinen trafolar - guc istatistikleri: 
 count    556319.000000
mean        683.975838
std        2172.301412
min          40.000000
25%         250.000000
50%         400.000000
75%        1000.000000
max       35900.000000
Name: guc, dtype: float64

Yeni trafolar - en sık lokasyonlar: 
 lokasyon
İZMİR>METROPOL>BORNOVA        17967
İZMİR>GÜNEY BÖLGE>MENDERES    10971
İZMİR>METROPOL>BAYRAKLI        9151
İZMİR>KUZEY BÖLGE>MENEMEN      7740
İZMİR>METROPOL>KARŞIYAKA       6066
İZMİR>METROPOL>KARABAĞLAR      5488
İZMİR>KUZEY BÖLGE>DİKİLİ       5293
İZMİR>GÜNEY BÖLGE>TORBALI      5151
İZMİR>METROPOL>BUCA            4745
İZMİR>METROPOL>KONAK           4531
MANİSA>YUNUSEMRE               4421
İZMİR>GÜNEY BÖLGE>ÖDEMİŞ       4313
MANİSA>SOMA         